# S-VLG — Huấn luyện V2 (SU-MedVQA) THẬT trên Colab GPU

Notebook này train model **thật** — Qwen2.5-3B-Instruct + QLoRA 4-bit — trên toàn bộ
VQA-RAD + SLAKE, khác với `scripts/run_smoketest_v2.py` chạy local trước đó (model tí hon
`tiny-gpt2` trên CPU, chỉ để kiểm chứng pipeline). Đây chính là bước tạo số liệu **final**
để điền vào bài báo khoa học.

## Bước 0 — Chuẩn bị (2 cách, chọn 1 qua biến `USE_DRIVE` ở cell dưới)

**Cách A — `USE_DRIVE = True` (mặc định, khuyến nghị nếu Drive còn đủ chỗ)**
1. Upload toàn bộ thư mục project `S-VLG` lên Google Drive (ví dụ `MyDrive/S-VLG`), gồm
   `data/raw/vqa-rad/` + `data/raw/slake/` (~280MB — code không cần upload lên Drive nữa,
   Bước 0.5 lấy thẳng từ GitHub). Checkpoint được tự động backup về Drive sau mỗi lần train.
2. KHÔNG cần upload `.git/`, `__pycache__/`, `outputs/checkpoints/`.

**Cách B — `USE_DRIVE = False` (Drive đã đầy)**
1. KHÔNG cần mount Drive, KHÔNG cần upload cả project — code tự `git clone` từ GitHub.
2. Bạn chỉ cần tự upload 2 thư mục `data/raw/vqa-rad/` + `data/raw/slake/` (~280MB) vào
   đúng chỗ trong Colab qua panel **Tệp** bên trái (kéo-thả), sau khi cell dưới chạy xong
   (nó sẽ tạo sẵn 2 thư mục rỗng cho bạn thả file vào).
3. QUAN TRỌNG: ổ đĩa Colab là **tạm thời** — mất hết khi ngắt kết nối/hết phiên. Không có
   Drive để tự động backup checkpoint, nên bạn phải **tự gọi `backup_outputs()`** định kỳ
   (nó sẽ nén `outputs/` và tự tải file zip về máy bạn) — xem Bước 2 và mục **Resume**.

Cả 2 cách: Runtime → Change runtime type → Hardware accelerator → GPU → chọn **A100**.

## Quy trình 3 bước
1. **Calibration**: chạy 1 epoch thật trên subset nhỏ để đo tốc độ THẬT trên A100 bạn đang dùng
   (số đo trên CPU/tiny-model KHÔNG dùng để suy ra thời gian GPU/Qwen được — kiến trúc decoder
   khác nhau hoàn toàn về quy mô).
2. **Full run**: toàn bộ VQA-RAD+SLAKE, N epoch, đủ 4 biến thể ablation.
3. **Compile**: gộp mọi số liệu vào `outputs/PAPER_DATA_v2.md` — file dùng để điền bài báo.

Checkpoint được lưu SAU MỖI EPOCH (chỉ giữ 3 bản gần nhất — mỗi checkpoint model thật ~4GB,
giữ hết 50 epoch sẽ tràn ổ đĩa). Nếu Colab ngắt kết nối giữa chừng, xem mục **Resume** ở cuối
notebook thay vì chạy lại từ đầu.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print("CUDA available:", torch.cuda.is_available())

In [ ]:
USE_DRIVE = True  # Đổi thành False nếu Google Drive đã đầy — xem Bước 0 (Cách B) phía trên.

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print("USE_DRIVE=False: bỏ qua mount Drive — code sẽ lấy từ GitHub, data tự upload thủ công.")

In [ ]:
# Dùng cho --backup-cmd: tự backup outputs/ (checkpoint + số liệu) sau MỖI epoch của model
# chính và sau MỖI biến thể ablation -- không cần chờ script chạy xong mới backup thủ công.
#
# QUAN TRỌNG (sự cố thật đã xảy ra): KHÔNG dùng --delete cho checkpoints/. Local tự dọn chỉ
# giữ 3 checkpoint gần nhất (keep-last-checkpoints) -- nếu backup cũng --delete mirror theo,
# một checkpoint TỐT ở epoch sớm (trước khi phát hiện loss phân kỳ) sẽ bị xoá khỏi CẢ local
# LẪN Drive ngay khi nó rớt khỏi cửa sổ 3 bản gần nhất, không còn cách nào phục hồi. Vì vậy:
# checkpoints/ luôn được ĐỒNG BỘ THÊM (không bao giờ xoá tự động) trên Drive -- Drive sẽ phình
# to dần theo số epoch thật đã train (không bị chặn bởi local pruning), bạn cần tự dọn thủ công
# trên Drive sau khi đã xác định xong checkpoint nào cần giữ (vd sau khi mark_final). Phần còn
# lại của outputs/ (tables/figures/PAPER_DATA_v2.md, dung lượng nhỏ) vẫn mirror --delete bình
# thường vì không có rủi ro mất dữ liệu quan trọng.
if USE_DRIVE:
    BACKUP_CMD = (
        f'rsync -a "{LOCAL_PROJECT_DIR}/outputs/checkpoints/" "{DRIVE_PROJECT_DIR}/outputs/checkpoints/" && '
        f'rsync -a --delete --exclude "checkpoints/" "{LOCAL_PROJECT_DIR}/outputs/" "{DRIVE_PROJECT_DIR}/outputs/"'
    )
else:
    BACKUP_CMD = None  # USE_DRIVE=False: không có đích tự động để backup mỗi epoch -- gọi
                        # backup_outputs() thủ công định kỳ thay vì tự động (xem Bước 2/3).
BACKUP_FLAG = f"--backup-cmd '{BACKUP_CMD}'" if BACKUP_CMD else ""


## Bước 0.5 — Đồng bộ CODE mới nhất từ GitHub (đè lên bản ở trên)

`data/raw/` (VQA-RAD + SLAKE) vẫn phải lấy thủ công như Bước 0 (Cách A: từ Drive: Cách B: tự
upload) — dữ liệu bị `.gitignore`, không nằm trên GitHub. Nhưng **code** (`src/`, `scripts/`,
`configs/`, `requirements.txt`) luôn lấy thẳng từ GitHub (`https://github.com/minhhaidhsp/s-vlg`)
để mỗi lần có bugfix bên máy local, bạn chỉ cần chạy lại cell dưới đây.

**Chỉ `checkout` đúng 4 thư mục/file code** (`src scripts configs requirements.txt`), KHÔNG
`reset --hard` toàn bộ cây — vì `outputs/tables/`, `outputs/figures/`, `outputs/PAPER_DATA_v2.md`
CÓ được track trong git (chứa số liệu pilot cũ từ máy local), `reset --hard` sẽ xóa mất kết quả
train thật của bạn nếu chạy lại cell này giữa/sau một lần train dài trên Colab.

In [ ]:
REPO_URL = "https://github.com/minhhaidhsp/s-vlg.git"

%cd $LOCAL_PROJECT_DIR
if not os.path.isdir(".git"):
    !git init -q
    !git remote add origin {REPO_URL}
!git fetch origin master -q

# Chỉ ghi đè CODE (src/scripts/configs/requirements.txt) từ GitHub — không đụng outputs/,
# data/, notes.txt hay bất kỳ file nào khác đang có trên đĩa Colab.
!git checkout origin/master -- src scripts configs requirements.txt
print("Code đã đồng bộ tới commit:")
!git log origin/master -1 --oneline

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
# Sanity check: dữ liệu và GPU đều sẵn sàng trước khi tốn thời gian train.
import json
from pathlib import Path

for name in ["vqa-rad", "slake"]:
    meta_path = Path(f"data/raw/{name}/metadata.json")
    assert meta_path.exists(), (
        f"Thiếu {meta_path} — kiểm tra lại Bước 0: USE_DRIVE=True thì xem đã upload lên Drive "
        f"đúng chỗ chưa; USE_DRIVE=False thì xem đã tự kéo-thả data/raw/{name}/ vào panel Tệp chưa."
    )
    data = json.loads(meta_path.read_text(encoding="utf-8"))
    print(f"{name}: {len(data)} cặp QA")

import torch
assert torch.cuda.is_available(), "KHÔNG có GPU — kiểm tra lại Runtime type (phải chọn GPU/A100)!"
print("GPU:", torch.cuda.get_device_name(0))

## Bước 1 — Calibration: đo thời gian thật cho 1 epoch

Chạy 1 epoch thật (model Qwen2.5+QLoRA thật, `--real`) trên subset nhỏ (200 mẫu/bộ, `--n 200`)
để đo tốc độ thật trên chính GPU bạn đang dùng. `--skip-ablation` bỏ qua 4 lần train ablation
(không cần cho calibration, tiết kiệm thời gian). Có progress bar theo batch.

In [ ]:
!python scripts/run_smoketest_v2.py --real --epochs 1 --n 200 --skip-ablation {BACKUP_FLAG}

## Bước 2 — Ước lượng thời gian full run

Đọc dòng `Trained 1 epoch(s) this run in Xs (Xs/epoch)` ở output cell trên.

Full dataset (VQA-RAD 2.244 + SLAKE 7.033 = 9.277 cặp QA) lớn hơn subset calibration
(1.000 cặp QA, `--n 200`×2) khoảng **9 lần**. Ước lượng thô cho N epoch:

```
thời gian ước tính ≈ (Xs/epoch đo được ở Bước 1) × 9 × N
```

Đây CHỈ là suy diễn tuyến tính đơn giản — thực tế A100 có thể tận dụng batch tốt hơn ở scale
lớn hơn (nhanh hơn ước lượng) hoặc bị giới hạn bởi I/O đọc ảnh (chậm hơn). Coi là tham khảo,
không phải cam kết. Cộng thêm ~4× thời gian 1 epoch nữa cho 4 biến thể ablation (mỗi biến thể
mặc định 1 epoch, xem `--ablation-epochs`).

In [ ]:
def backup_outputs():
    """Backup outputs/ (checkpoints, tables, PAPER_DATA_v2.md) đề phòng mất khi Colab ngắt.
    USE_DRIVE=True: đồng bộ về Google Drive (tự động, im lặng, khuyến nghị).
    USE_DRIVE=False: nén outputs/ thành zip và TỰ TẢI VỀ máy bạn qua trình duyệt — vì không có
    Drive, BẠN PHẢI tự gọi hàm này định kỳ (vd sau mỗi vài epoch của lần train dài) và giữ lại
    file zip mới nhất — xem mục Resume để biết cách khôi phục từ file zip đó."""
    if USE_DRIVE:
        !rsync -a "$LOCAL_PROJECT_DIR"/outputs/ "$DRIVE_PROJECT_DIR"/outputs/
        print("Đã đồng bộ outputs/ về Google Drive.")
    else:
        import shutil
        from google.colab import files
        zip_path = "/content/s-vlg-outputs-backup"
        shutil.make_archive(zip_path, "zip", f"{LOCAL_PROJECT_DIR}/outputs")
        print(f"Đã nén outputs/ -> {zip_path}.zip — trình duyệt sẽ hỏi tải file về, hãy lưu lại.")
        files.download(f"{zip_path}.zip")


backup_outputs()

## Bước 3 — Chạy FULL: toàn bộ dataset, N epoch, đủ 4 biến thể ablation

- `--n 0`: dùng TOÀN BỘ VQA-RAD + SLAKE (không lấy subset).
- `EPOCHS`: NGƯỠNG TRẦN — sửa theo ước lượng ở Bước 2 và budget thời gian bạn có (đề xuất 10-50).
- **Early stopping** (`EARLY_STOP_PATIENCE`): sau MỖI epoch, script tự eval trên tập VAL (không phải
  test, tránh rò rỉ số liệu cuối). Nếu `vqa_acc` trên val không cải thiện sau N epoch liên tiếp
  (N = `EARLY_STOP_PATIENCE`), DỪNG SỚM thay vì chạy hết `EPOCHS` — Bảng 6/7/10/11 sẽ dùng đúng
  checkpoint của epoch TỐT NHẤT (không phải epoch cuối lúc dừng). Đặt `EARLY_STOP_PATIENCE = None`
  để tắt (chạy đúng hết `EPOCHS`, hành vi cũ). LƯU Ý: eval mỗi epoch tốn thêm thời gian thật (có
  `generate()` cho câu OPEN) — đây là chi phí vốn có của early stopping, không tránh được.
  KHÔNG được phép dừng trước khi KL annealing xong (`kl_warmup_epochs`, mặc định 10 epoch) — trong giai đoạn đó hàm mục tiêu vẫn đang đổi (beta_t tăng dần), val accuracy có thể chững tạm thời không phản ánh hội tụ thật. Vẫn theo dõi/lưu best checkpoint từ epoch 1.
- Checkpoint lưu SAU MỖI EPOCH vào `outputs/checkpoints/v2_real/` (chỉ giữ 3 bản gần nhất, RIÊNG
  checkpoint epoch tốt nhất `*_best.pt` không bao giờ bị dọn).
- **Backup tự động** (`BACKUP_FLAG`, tính từ `USE_DRIVE` ở Bước 0): nếu `USE_DRIVE=True`,
  script tự rsync `outputs/` về Drive SAU MỖI EPOCH của model chính (kèm mỗi lần có epoch tốt
  nhất mới), SAU KHI model chính train xong (tách biệt với ablation), và SAU MỖI biến thể ablation
  riêng lẻ — nếu Colab ngắt kết nối giữa chừng, bạn chỉ mất tối đa phần đang chạy dở. Không cần tự
  gọi `backup_outputs()` giữa chừng nữa.
- **Mỗi lần chạy script sẽ THÊM bản ghi mới vào `outputs/tables/v2/*.json`, không ghi đè** — nếu
  bạn đã chạy calibration hoặc chạy thử ở trên với `--n 200`, các bảng số liệu sẽ có thêm dòng
  đó. `compile_paper_data.py` (Bước 4) tự lấy bản ghi MỚI NHẤT cho mỗi model/dataset khi không
  đủ ≥2 seed để tính mean±std, nên số liệu cuối cùng vẫn đúng — nhưng nếu muốn bảng sạch hoàn
  toàn chỉ có 1 lần chạy thật, hãy xoá `outputs/tables/v2/` và `outputs/figures/data/v2/` trên
  Drive trước khi chạy Bước 3.

⚠️ Đây là lần chạy dài (có thể vài chục phút đến vài giờ tuỳ EPOCHS) — giữ tab Colab mở.

In [ ]:
EPOCHS = 10  # sửa nếu muốn khác, dựa trên ước lượng ở Bước 2
EARLY_STOP_PATIENCE = 5  # None để tắt (chạy đúng hết EPOCHS); vd 5 = dừng nếu 5 epoch liên tiếp val không cải thiện

EARLY_STOP_FLAG = f"--early-stop-patience {EARLY_STOP_PATIENCE}" if EARLY_STOP_PATIENCE is not None else ""

!python scripts/run_smoketest_v2.py --real --epochs {EPOCHS} --n 0 {BACKUP_FLAG} {EARLY_STOP_FLAG}

backup_outputs()  # backup cuối cùng, phòng khi BACKUP_CMD=None (USE_DRIVE=False)

## Bước 4 — Xem kết quả

`compile_paper_data.py` đã tự chạy ở cuối Bước 3 (là bước cuối của `run_smoketest_v2.py`).
In lại nội dung `outputs/PAPER_DATA_v2.md` ở đây cho tiện đọc/copy.

In [ ]:
print(Path("outputs/PAPER_DATA_v2.md").read_text(encoding="utf-8"))

## Resume — nếu Colab ngắt kết nối giữa chừng Bước 3

**Nếu `USE_DRIVE = True`:**
1. Chạy lại từ đầu notebook: mount Drive → copy project về local (cell ở Bước 0) — vì không
   exclude `outputs/checkpoints` nên checkpoint đã đồng bộ lên Drive lần trước (qua
   `backup_outputs()`) sẽ được copy về local.
2. Chạy 2 cell dưới đây thay cho Bước 3.

**Nếu `USE_DRIVE = False`:**
1. Chạy lại Bước 0 (sẽ `git clone` code mới, tạo lại thư mục `data/raw/` rỗng) — upload lại
   `data/raw/vqa-rad/` + `data/raw/slake/` như lần đầu.
2. Upload file zip backup gần nhất (tải về lúc trước từ `backup_outputs()`) vào Colab qua
   panel Tệp, rồi chạy cell "Giải nén backup" bên dưới trước khi chạy 2 cell Resume.

Lưu ý chung: `--resume-from` chỉ khôi phục việc train model chính (Bước b); nếu bị ngắt trong
lúc chạy ablation (Bước d), script sẽ chạy lại cả 4 biến thể ablation từ đầu khi rerun (mỗi
biến thể chỉ 1 epoch nên chi phí chạy lại không lớn).

In [ ]:
# Chỉ cần chạy cell này nếu USE_DRIVE=False: giải nén file zip backup (upload thủ công vào
# /content qua panel Tệp) đè vào outputs/, trước khi 2 cell Resume bên dưới tìm checkpoint.
if not USE_DRIVE:
    import shutil
    from pathlib import Path

    zip_candidates = sorted(Path("/content").glob("*.zip"))
    print("File zip tìm thấy trong /content:", zip_candidates)
    assert zip_candidates, "Chưa thấy file zip backup nào — upload file backup_outputs() đã tải về trước đó vào /content."
    RESTORE_ZIP = str(zip_candidates[-1])
    print("Giải nén:", RESTORE_ZIP)
    shutil.unpack_archive(RESTORE_ZIP, f"{LOCAL_PROJECT_DIR}/outputs")

In [ ]:
import glob

ckpts = sorted(glob.glob(f"{LOCAL_PROJECT_DIR}/outputs/checkpoints/v2_real/full/*.pt"))
print("Checkpoint tìm thấy:", ckpts)
RESUME_FROM = ckpts[-1] if ckpts else None
print("Sẽ resume từ:", RESUME_FROM)
assert RESUME_FROM is not None, "Không tìm thấy checkpoint nào — có thể Bước 3 chưa lưu được epoch nào, chạy lại Bước 3 bình thường."

In [ ]:
# EPOCHS phải GIỮ NGUYÊN mục tiêu tổng như lần chạy trước (vd 10) — script tự tính số epoch
# còn thiếu (nếu checkpoint đã ở epoch 6/10 thì chỉ train tiếp 7..10, không train lại từ đầu).
# LƯU Ý: early stopping (nếu bật) KHÔNG nhớ được best/patience của lần chạy trước khi resume —
# nó bắt đầu đếm lại từ epoch resume. Nếu lần trước đã có best tốt ở epoch sớm hơn, file
# *_best.pt cũ đó vẫn còn nguyên trên đĩa (không bị resume ghi đè cho tới khi có best MỚI thật).
!python scripts/run_smoketest_v2.py --real --epochs {EPOCHS} --n 0 --resume-from "{RESUME_FROM}" {BACKUP_FLAG} {EARLY_STOP_FLAG}

backup_outputs()

## Sau khi hài lòng với số liệu: đánh dấu FINAL

Mọi số liệu ghi ra đều có `status="provisional"`. Khi đã chạy đủ epoch/seed bạn muốn dùng làm
số chính thức cho bài báo, chạy đoạn dưới (đổi tham số cho đúng bảng/model/seed) để nâng cấp
lên `"final"` — khi đó `compile_paper_data.py` sẽ in nhãn `[FINAL]` thay vì `[TẠM epoch=N]`.

In [ ]:
from src.utils.results_logger import ResultsLogger

logger = ResultsLogger(experiment_version="v2")
logger.mark_final("table6_overall", key_value="SU-MedVQA", key_field="model_name", dataset="vqa-rad+slake")
# Lặp lại cho các bảng khác nếu cần: table7_by_category, table9_ablation (key_field="variant_name"),
# table10_risk_coverage (key_field="config_name"), table11_efficiency.

!python scripts/compile_paper_data.py --version v2
backup_outputs()